Importing required libraries

In [ ]:
import aner.pca_ranking as pca_ranking
import aner.AnerLib as AnerLib
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from collections import defaultdict


Setting parameters: constants, tranining set and fileName

In [ ]:
trainingSet = ['ccnb1', 'tpx2', 'aurka', 'cdc20', 'ccna2']
#trainingSet = ["mos", "cdc6", "slbp", "prc1", "btg4", "cnot7", "cnot8"]


## nearest neighbors that the algorithm will consider (0 = will consider all genes)
NN = 100
top = 30

dataset = "../examples/Mitotic_datasets.csv"
#dataset = "../examples/Oocyte_datasets.csv"


Load data and Data scalling

In [ ]:
#Load data
data = pd.read_csv(dataset, index_col=0, header=0)
arr = data

# convert gene index to list
geneIndex = data.index.to_list()

#Data scalling
scaler = StandardScaler()
X_normalized = scaler.fit_transform(arr)

# Getting the index and value of the training set genes
trainingSetIndex = [geneIndex.index(gene) for gene in trainingSet]
trainingSetValue = [arr.iloc[idx] for idx in trainingSetIndex]

Ranking with PCA strategy: getting closest neighbor of trainingSet using  PCA coordinates

In [ ]:
neighborPCA = pca_ranking.getClosestPCA(top, trainingSetIndex, X_normalized)
topranked = pca_ranking.saveTop(neighborPCA, top, geneIndex, "", "\t")

Printing the top ranked

In [ ]:
print(topranked)

Ranking with ANeR

In [ ]:
 # Aner
G, adj = AnerLib.processGraphAner(arr, trainingSetIndex, geneIndex, 'euclidean', True)
rankMat, rankMat2 = AnerLib.rankingNodes(G, trainingSetIndex)
rankMat.sort(key=lambda k: (k[1], -k[4], k[5], k[2], k[3]), reverse=True)

Printing top ranked

In [ ]:
topranked = AnerLib.printRank(rankMat,geneIndex,top,"\t")
print(topranked)


Applying Pareto frontiers

In [ ]:
pareto_final = AnerLib.getParetoSet(rankMat2)
list_complet_genes, list_complete = AnerLib.multiObjRanking(pareto_final, rankMat)
topRankedPareto = AnerLib.printFrontPareto(list_complete, geneIndex, trainingSetIndex, top)

print(topRankedPareto)
